CNN IN PYTORCH SU CIFAR-10: GESTIONE DEL COLORE E MONITORAGGIO AVANZATO

Segnale cromatrico, esploriamo come le macchine interpretano la profondità del colore.

CIFRA-10 è un dataset di immagini piccole, 32x32 a colori, divise in 10 classi (aereo, auto, uccello, gatto, cervo, cane, rana, cavallo, nave, camion)
Mentre MNIST è grigio, CIFAR-10 è a colori

Un'immagine in CIFRA-10 ha forma
(3x32x32)
(canali, altezza, larghezza)

Ogni immagine ha 3 'strati': Rosso, Verde, Blu. Un pixel non ha un solo colore, ma tre valori pixel=[R,G,B]
Esempio:
rosso=[255,0,0]
verde=[0,255,0]
blu=[0,0,255]
bianco=[255,255,255]
nero=[0,0,0]
Quando carichi CIFAR-10 con PyTorch, normalmente trasformi questi valori da 0-255 a valori circa tra 0 e 1 usando ToTensor()
Poi normalizzi ogni canale colore
transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),std=(0.2470, 0.2435, 0.2616))
Perchè il colore cambia tutto?

La Complessità del Colore
Il passaggio dai singoli canali alla profondità RGB
Mentre con MNIST abbiamo vissuto in un mondo bidimensionale, ogni pixel era solo un ombra grigio e ci ha permesso di studiare la geometria delle forme in bianco e nero, il dataset CIFAR-10 ci introduce alla relatà del segnale cromatico. Ogni pixel non è più un singolo valore, ma un vettore di tra intensità (valori): Rosso Verde Blu.
Questa transizione richiede che i nostri filtri convoluzionali non scorrano solo su una superficie piana, ma operino su un valume catturando le correlazioni tra i diversi canali colore per identificare pattern complessi.

Tensolri a Tre Canali
La struttura spaziale delle immagini CIFAR-10
Risoluzione e profondità: le immagini CIFAR10 sono matrici 32x32 con 3 canali. generano un volume in input di 3072 valori totali per ogni singola immagine.
Pytorch è pignolo sull'ordine delle dimensioni, mentri altri framework preferiscono avere i canali alla fine, PyTorch vuole i canali all'inizion
Batch + canali + altezza + larghezza.

Canali e Friltri. 
Poichè l'input ha profondità 3 anche i filtri del nostri primo leyer devono avere profondità 3. Questo significa triplicare i calcoli rispetto a MINST
Per gestire questo carico senza far esplodere i gradienti, useremo la normalizzazione per canale.,
Una volta normalizzati i colori, la rete inizia a vedere la semantica.

Gerarchia del Colore.
Dal pixel alla semantica cromatica.
Nelle fasi inziali, la rete può specializzarsi nel rilevare contrasti di colore (es. il giallo di un becco il blu dell'acquia). Questa capacità (es. il giallo di un becco contro il blu dell'acqua). Questa capacità è preclusa ai modelli che lavorano solo in scale di grigio.

In PyTorch il caricamento del dato non è un evento statico ma una pipeline dinamica. Utilizzeremo il modulo 'transforms' per preparare le immagini prima che queste vengano somministrate al modello.
Questa fase è cruciale per la generalizzazione: una corretta trasformazione impedisce alla rete di memorizzare pixel rumorosi e favorisce l'apprendimento di feature robuste.
Se prepariamo bene i dati qui, la rete sarà molto più brava a riconoscere un oggetto anche con una foto sgrananta.

Standarizzazione del Flusso
Converitre immagini in tensori pronti al trainig
- ToTensor: è il nostro traduttore, trasforma l'immagine PIL o array NumPy (leggibile dall'uomo) in un tensore di float e scalla i valori nell'intervallo 0,1
- Z-Score Normalization: sottraendo la media e dividendo per la deviazione stanrda, rendiamo il gradiente più stabile e veloce, centrando i dati attorno allo zero.
Composizione: transforms.Compose permette di concatenare 
L'operazione di normalizzazoine trasforma ogni canale sottranendo la media specifica e dividendo per lo scarto quadratico medio.
Ma la pipelne può fare molto di più di una semplice pulizia.

Efficienza della Pipeline
Possiamo rendere la nostra rete pià furba aggiungendo piccoli trucchi.
Data Augmentation di Base: Sebbene l'augmentation verrà trattata in seguito, piccoli accorgimenti come il 'RandomHorizontalFlip' possono già migliorare la robustezza del modello.
Gestione Batch: Il DataLoader organizza le trasformazioni in parallelo usando diversi 'worker', ottimizzando i tempi di attesa tra un'epoca e l'altra.
Mapping delle Classi: CIFAR-10 utilizza indici interi, è compito della pipeline mappare correttamente questi indici ai nomi delle classi leggibili per l'utente finale.

Il Ciclo di Caricamento
Dal disco alla GPU
A differenza di Keras dove i dati sono caricati tutti in memoria RAM, PyTorch usa gli iteratori. Questo approccio permette di gestire dataset che superano la capacità della RAM fisica del sistema.
La trasformazione avviene 'on-the-fly' durante l'iterazione (solo quando servono), garantendo che il processore grafico riceva sempre dati pronti per il calcolo dei gradienti.
Ora che la rete sta imparando come facciamo a sapere se sta capendo tutto?

Diagnostica e Monitoraggio
Analisi fine dell'apprendimento per categoria
Un errore comune è guardare solo l'accuratezza globale. Tuttavia, un modello potrebbe essere eccellente nel riconoscere navi ma fallire miseramente con i gatti, portando a una media ingannevole.
Ci vuole un sistema di monitoraggio che traccia le performance specifica per ogni classe, permettendoci di identificare eventuali confusioni semantiche del modello.

Loss per Classe
Identificare i punti deboli del classificatore.
Contare successi e fallimenti per ogni singola etichetta
- Confusion Matrix Implicita: monitorare quali classi vengono scambiate più frequentemente aiuta a capire se la rete manca di dettagli discriminanti.
- Valutazione per Classe: calcolando il rapporto tra predizoini correte e totali per ogni etichetta, otteniamo una vista granulare della salute del modello.
- Accumulatori di Statistiche: durante il test, memorizzando separatamente i risuotati per le 10 classi di CIFAR10 per generare un report finale di precisione
- L'accuretezza per la classe i-esima è il rapporto tra i veri positivi e il totale degli esempi appartenenti a quella categoria.

Poi per correggere questi errori dobbiamo utilizzare funzioni di costo giuste.

Techinche di Monitoraggio
Utilizzeremo la nn.CrossEntropyLoss che combina internamente LogSoftmax e NLLLoss, ideale per problemi multiclasse come CIFAR10
Possiamo utilizzare l'ottimizzatorei SGD con momentum per navigare il paesaggio della loss più accidentato rispetto a quello di MNIST (oppure un Adam più veloce che agisce come una pallina che rotola giù da un colle).
La perdita misurata sul set di validazione alla fine di ogni epoca, permette di prevenire l'overfitting e decidere quando interrompere l'addestramento.

Interpretazione degli errori.
Perchè la rete confonde cani e gatti?
Il report per classe permette di scoprire oggetti con strutture simili (come gatti e cani) e che tendono ad avere accuratezze inferiore rispetto a classi molto diverse (come navi e aerei)
Questa analisi ci giuderà nella futura implementazione di tecniche di regolarizzazione più agressive o nell'aumento della profondità dei layer convoluzionali.


In [ ]:
#AI Overview
#TensorFlow/Keras codice copre CNN, RNN, Adam, e Multi-Backend Keras 3 per Deep Learning avanzato.

import os

# 1. CONFIGURAZIONE BACKEND (Best Practice 2026)
# Keras 3 permette di scegliere il motore di calcolo (backend). 
# Impostiamo "torch" prima di importare keras per sfruttare l'ecosistema PyTorch.
# Teoria: L'agnosticismo del backend permette di addestrare su un framework e distribuire su un altro.
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers, models, ops
import numpy as np
import tensorflow as tf # Utilizzato esclusivamente per la pipeline tf.data (standard industriale per prefetch)
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# 2. PIPELINE DATI OTTIMIZZATA (Prefetch & Cache)
# ==========================================

def prepare_dataset(x, y, training=True):
    """
    Crea una pipeline di dati performante.
    Teoria: Il collo di bottiglia nel DL è spesso il caricamento dati (CPU) rispetto al calcolo (GPU).
    """
    # Creiamo un oggetto Dataset dai tensori NumPy
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    
    # [TEORIA] Normalizzazione: Portiamo i pixel da [0, 255] a [0, 1].
    # Input piccoli e centrati evitano che i gradienti esplodano durante le prime epoche.
    dataset = dataset.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), 
                          num_parallel_calls=tf.data.AUTOTUNE)
    
    # [CACHE] Memorizza i dati in RAM dopo la prima lettura.
    # Evita di ripetere le operazioni di decodifica/preprocessing ad ogni epoca.
    dataset = dataset.cache()
    
    if training:
        # [SHUFFLE] Fondamentale per la convergenza: evita che il modello impari l'ordine dei dati.
        dataset = dataset.shuffle(buffer_size=5000)
    
    # Batching: raggruppiamo i dati.
    dataset = dataset.batch(64)
    
    # [PREFETCH] La "Killer Feature" per le performance.
    # Mentre la GPU calcola il gradiente del batch corrente (N), 
    # la CPU prepara già il batch successivo (N+1) in background.
    # AUTOTUNE decide dinamicamente quanti batch precaricare in base alle risorse.
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    
    return dataset

# Caricamento dati CIFAR-10
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras.datasets.cifar10.load_data()

# Preparazione pipeline
train_ds = prepare_dataset(x_train_raw, y_train_raw, training=True)
test_ds = prepare_dataset(x_test_raw, y_test_raw, training=False)

# ==========================================
# 3. ARCHITETTURA: RESNET CUSTOM (API Funzionale)
# ==========================================

def residual_block(x, filters, stride=1):
    """
    Blocco Residuale (ResNet). 
    Teoria: Introduce le 'skip connections' per risolvere il problema della scomparsa del gradiente.
    Permette ai gradienti di fluire direttamente attraverso la rete durante la backpropagation.
    """
    shortcut = x
    
    # Percorso principale - Primo blocco: Conv -> BN -> Activation
    # La Batch Normalization stabilizza l'apprendimento normalizzando le attivazioni medie.
    x = layers.Conv2D(filters, 3, strides=stride, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    # Secondo blocco: Conv -> BN
    x = layers.Conv2D(filters, 3, strides=1, padding="same")(x)
    x = layers.BatchNormalization()(x)
    
    # [MATCHING DIMENSIONI] Se cambiamo risoluzione (stride > 1), dobbiamo adattare anche lo shortcut
    # in modo che la somma element-wise sia matematicamente possibile.
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    # [TEORIA] H(x) = F(x) + x. Aggiungiamo l'input originale all'output trasformato.
    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)
    return x

def build_model(input_shape=(32, 32, 3), num_classes=10):
    inputs = layers.Input(shape=input_shape)
    
    # Layer iniziale di estrazione feature (Stem)
    x = layers.Conv2D(32, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    # Stack di blocchi residuali con aumento progressivo dei filtri
    x = residual_block(x, 64)
    x = residual_block(x, 128, stride=2) # Dimezza risoluzione (16x16)
    x = residual_block(x, 256, stride=2) # Dimezza risoluzione (8x8)
    
    # [GLOBAL AVERAGE POOLING] Best practice 2026 al posto del Flatten().
    # Riduce il numero di parametri e rende il modello più robusto alle traslazioni spaziali.
    x = layers.GlobalAveragePooling2D()(x)
    
    # Output layer con Softmax per classificazione multi-classe.
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    
    return models.Model(inputs, outputs, name="ResNet_CIFAR10_2026")

model = build_model()
model.summary()

# ==========================================
# 4. COMPILAZIONE E TRAINING
# ==========================================

# [ADAMW] Optimizer standard nel 2026: Adam con Weight Decay integrato per una migliore regolarizzazione.
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# [CALLBACKS] Automazione del training
callbacks = [
    # Riduce il LR se la loss smette di migliorare (ottimizzazione fine)
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3),
    # Ferma il training per evitare l'overfitting
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

print("\n[INFO] Avvio addestramento con backend PyTorch...")
history = model.fit(
    train_ds, 
    validation_data=test_ds, 
    epochs=10, 
    callbacks=callbacks
)

# ==========================================
# 5. SALVATAGGIO MODELLO (Formato 2026)
# ==========================================

# Il formato .keras è lo standard V3: contiene pesi, architettura e stato dell'ottimizzatore.
# È un file compresso e platform-independent.
model.save("cifar10_model_v2026.keras")
print(f"\n[INFO] Modello salvato con successo: cifar10_model_v2026.keras")

# ==========================================
# 6. VALUTAZIONE E ANALISI
# ==========================================

# Generazione predizioni
y_pred_all = []
y_true_all = []

for x_batch, y_batch in test_ds:
    preds = model.predict(x_batch, verbose=0)
    y_pred_all.extend(np.argmax(preds, axis=1))
    y_true_all.extend(y_batch.numpy())

# Report di classificazione
target_names = ['Aereo', 'Auto', 'Uccello', 'Gatto', 'Cervo', 'Cane', 'Rana', 'Cavallo', 'Nave', 'Camion']
print("\n--- PERFORMANCE SUL TEST SET ---")
print(classification_report(y_true_all, y_pred_all, target_names=target_names))

# Matrice di Confusione Visiva
cm = confusion_matrix(y_true_all, y_pred_all)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Analisi Errori (Confusion Matrix)")
plt.colorbar()
plt.xlabel("Predetto")
plt.ylabel("Reale")
plt.show()

Model: "ResNet_CIFAR10_2026"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │     18,496 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     36,928 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │      2,112 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 32, 32,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │     73,856 │ activation_2[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        512 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 16, 16,    │    147,584 │ activation_3[0][

 Total params: 1,214,538 (4.63 MB)

 Trainable params: 1,211,786 (4.62 MB)

 Non-trainable params: 2,752 (10.75 KB)


[INFO] Avvio addestramento con backend PyTorch...
Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 1181s 2s/step - accuracy: 0.5575 - loss: 1.2269 - val_accuracy: 0.5429 - val_loss: 1.3552 - learning_rate: 0.0010
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 1196s 2s/step - accuracy: 0.7056 - loss: 0.8324 - val_accuracy: 0.5092 - val_loss: 1.6816 - learning_rate: 0.0010
Epoch 3/10
744/782 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step - accuracy: 0.7538 - loss: 0.6981